<a href="https://colab.research.google.com/github/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/blob/main/ACE6313_Group_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- PART A: DATA PREPROCESSING By Diaa ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Starting Part A: Data Preprocessing Pipeline...\n" + "-"*40)

# ==========================================
# 1. DATA CLEANING
# ==========================================
url = 'https://raw.githubusercontent.com/DiaaEddin963/ACE6313-Employee-Attrition-Group-I/main/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(url)

# A. Handle Missing Values & Duplicates
df = df.dropna().drop_duplicates()

# B. Handle Inconsistencies (Drop uninformative features)
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')

# C. Handle Outliers (Using IQR method to cap extreme Monthly Incomes)
Q1 = df_cleaned['MonthlyIncome'].quantile(0.25)
Q3 = df_cleaned['MonthlyIncome'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
# Cap outliers instead of dropping them to preserve valuable minority class data
df_cleaned.loc[df_cleaned['MonthlyIncome'] > upper_bound, 'MonthlyIncome'] = upper_bound

print(f"1. Data Cleaning Complete.")
print(f"   -> Handled missing values, duplicates, inconsistencies, and capped Outliers.")
print(f"   -> Current Shape: {df_cleaned.shape}")

# ==========================================
# 2. DATA TRANSFORMATION
# ==========================================
# A. Feature Engineering (Creating a meaningful new metric)
# Creating a 'Tenure-to-Age Ratio' to see if employees who spent their whole youth here leave less
df_cleaned['Tenure_Age_Ratio'] = df_cleaned['YearsAtCompany'] / df_cleaned['Age']

# B. Target Separation
y = df_cleaned['Attrition'].map({'Yes': 1, 'No': 0})
X = df_cleaned.drop('Attrition', axis=1)

# C. Feature Encoding (Categorical to Binary)
X_encoded = pd.get_dummies(X, drop_first=True)

# D. Feature Scaling (Standardizing numerical ranges)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_transformed = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print(f"\n2. Data Transformation Complete.")
print(f"   -> Performed Feature Engineering, One-Hot Encoding, and Standard Scaling.")
print(f"   -> Current Shape: {X_transformed.shape}")

# ==========================================
# 3. DATA REDUCTION (Feature Selection)
# ==========================================
from sklearn.feature_selection import SelectKBest, f_classif

# Select the top 20 most predictive features instead of using PCA
# This preserves feature meaning and tree-based model performance
selector = SelectKBest(score_func=f_classif, k=20)
X_reduced = selector.fit_transform(X_scaled, y)

# Keep the data as a DataFrame to retain column names for HR interpretation
selected_columns = X_transformed.columns[selector.get_support()]
X_final = pd.DataFrame(X_reduced, columns=selected_columns)

print(f"\n3. Data Reduction Complete.")
print(f"   -> Applied Feature Selection (SelectKBest).")
print(f"   -> Reduced features from {X_transformed.shape[1]} down to {X_final.shape[1]} most critical features.")
print("-" * 40 + "\nPART A FULLY COMPLETED. DATA IS READY FOR PART B.")

Starting Part A: Data Preprocessing Pipeline...
----------------------------------------
1. Data Cleaning Complete.
   -> Handled missing values, duplicates, inconsistencies, and capped Outliers.
   -> Current Shape: (1470, 31)

2. Data Transformation Complete.
   -> Performed Feature Engineering, One-Hot Encoding, and Standard Scaling.
   -> Current Shape: (1470, 45)

3. Data Reduction Complete.
   -> Applied Feature Selection (SelectKBest).
   -> Reduced features from 45 down to 20 most critical features.
----------------------------------------
PART A FULLY COMPLETED. DATA IS READY FOR PART B.


In [ ]:
# --- CELL 1: PREPARE BOTH DATASETS ---
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
import pandas as pd

print("--- PREPARING A/B TEST DATA ---")

# 1. Recreate X_pca (Original Method)
# We recreate it here just in case Diaa's previous cell deleted it
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_transformed)

# 2. X_final already exists from Diaa's latest code (SelectKBest)
# X_final = ... (already in memory)

# 3. Create Two Separate Train/Test Splits
# We use the exact same random_state (42) so the same employees end up in the test set
X_train_pca, X_test_pca, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

X_train_final, X_test_final, _, _ = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print(f"PCA Training set size: {X_train_pca.shape}")
print(f"Final (SelectKBest) Training set size: {X_train_final.shape}")
print("Ready for Model Comparison!")

--- PREPARING A/B TEST DATA ---
PCA Training set size: (1176, 28)
Final (SelectKBest) Training set size: (1176, 20)
Ready for Model Comparison!


In [ ]:
# --- 1. LOGISTIC REGRESSION A/B TEST ---
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("--- 1. LOGISTIC REGRESSION ---")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_params = {"C": [0.1, 1, 10], "solver": ["liblinear"]}

# 1. Train on PCA
grid_pca = GridSearchCV(lr, lr_params, cv=5, scoring='f1', n_jobs=-1)
grid_pca.fit(X_train_pca, y_train)
rep_pca = classification_report(y_test, grid_pca.predict(X_test_pca), output_dict=True)

# 2. Train on Final (SelectKBest)
grid_final = GridSearchCV(lr, lr_params, cv=5, scoring='f1', n_jobs=-1)
grid_final.fit(X_train_final, y_train)
rep_final = classification_report(y_test, grid_final.predict(X_test_final), output_dict=True)

# 3. Print Results
print(f"Best PCA Params:   {grid_pca.best_params_}")
print(f"Best Final Params: {grid_final.best_params_}\n")
print(f"{'Dataset':<15} | {'Accuracy':<8} | {'Class 1 Precision':<18} | {'Class 1 Recall':<15} | {'Class 1 F1'}")
print("-" * 80)
print(f"{'PCA':<15} | {rep_pca['accuracy']:.2f}     | {rep_pca['1']['precision']:.2f}               | {rep_pca['1']['recall']:.2f}            | {rep_pca['1']['f1-score']:.2f}")
print(f"{'SelectKBest':<15} | {rep_final['accuracy']:.2f}     | {rep_final['1']['precision']:.2f}               | {rep_final['1']['recall']:.2f}            | {rep_final['1']['f1-score']:.2f}")


--- 1. LOGISTIC REGRESSION ---
Best PCA Params:   {'C': 10, 'solver': 'liblinear'}
Best Final Params: {'C': 1, 'solver': 'liblinear'}

Dataset         | Accuracy | Class 1 Precision  | Class 1 Recall  | Class 1 F1
--------------------------------------------------------------------------------
PCA             | 0.78     | 0.39               | 0.68            | 0.49
SelectKBest     | 0.76     | 0.36               | 0.66            | 0.47


In [ ]:
# --- 2. K-NEAREST NEIGHBORS A/B TEST (With Custom Threshold) ---
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import numpy as np

print("--- 2. K-NEAREST NEIGHBORS (Threshold 0.2) ---")
knn = KNeighborsClassifier()
knn_params = {"n_neighbors": [3, 5, 7], "weights": ["uniform", "distance"]}
THRESHOLD = 0.2

# 1. Train on PCA
grid_pca = GridSearchCV(knn, knn_params, cv=5, scoring='f1', n_jobs=-1)
grid_pca.fit(X_train_pca, y_train)
y_proba_pca = grid_pca.predict_proba(X_test_pca)[:, 1]
y_pred_pca = (y_proba_pca >= THRESHOLD).astype(int)
rep_pca = classification_report(y_test, y_pred_pca, output_dict=True)

# 2. Train on Final (SelectKBest)
grid_final = GridSearchCV(knn, knn_params, cv=5, scoring='f1', n_jobs=-1)
grid_final.fit(X_train_final, y_train)
y_proba_final = grid_final.predict_proba(X_test_final)[:, 1]
y_pred_final = (y_proba_final >= THRESHOLD).astype(int)
rep_final = classification_report(y_test, y_pred_final, output_dict=True)

# 3. Print Results
print(f"Best PCA Params:   {grid_pca.best_params_}")
print(f"Best Final Params: {grid_final.best_params_}\n")
print(f"{'Dataset':<15} | {'Accuracy':<8} | {'Class 1 Precision':<18} | {'Class 1 Recall':<15} | {'Class 1 F1'}")
print("-" * 80)
print(f"{'PCA':<15} | {rep_pca['accuracy']:.2f}     | {rep_pca['1']['precision']:.2f}               | {rep_pca['1']['recall']:.2f}            | {rep_pca['1']['f1-score']:.2f}")
print(f"{'SelectKBest':<15} | {rep_final['accuracy']:.2f}     | {rep_final['1']['precision']:.2f}               | {rep_final['1']['recall']:.2f}            | {rep_final['1']['f1-score']:.2f}")

--- 5. K-NEAREST NEIGHBORS (Threshold 0.2) ---
Best PCA Params:   {'n_neighbors': 3, 'weights': 'uniform'}
Best Final Params: {'n_neighbors': 3, 'weights': 'distance'}

Dataset         | Accuracy | Class 1 Precision  | Class 1 Recall  | Class 1 F1
--------------------------------------------------------------------------------
PCA             | 0.69     | 0.24               | 0.43            | 0.31
SelectKBest     | 0.73     | 0.29               | 0.49            | 0.37


In [ ]:
# --- 3. SUPPORT VECTOR MACHINE A/B TEST ---
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report

print("--- 3. SUPPORT VECTOR MACHINE ---")
svm = SVC(probability=True, random_state=42)
svm_params = {"C": [1, 10, 50], "kernel": ["rbf"], "class_weight": ['balanced', {0: 1, 1: 3}]}

# 1. Train on PCA
grid_pca = GridSearchCV(svm, svm_params, cv=5, scoring='f1', n_jobs=-1)
grid_pca.fit(X_train_pca, y_train)
rep_pca = classification_report(y_test, grid_pca.predict(X_test_pca), output_dict=True)

# 2. Train on Final (SelectKBest)
grid_final = GridSearchCV(svm, svm_params, cv=5, scoring='f1', n_jobs=-1)
grid_final.fit(X_train_final, y_train)
rep_final = classification_report(y_test, grid_final.predict(X_test_final), output_dict=True)

# 3. Print Results
print(f"Best PCA Params:   {grid_pca.best_params_}")
print(f"Best Final Params: {grid_final.best_params_}\n")
print(f"{'Dataset':<15} | {'Accuracy':<8} | {'Class 1 Precision':<18} | {'Class 1 Recall':<15} | {'Class 1 F1'}")
print("-" * 80)
print(f"{'PCA':<15} | {rep_pca['accuracy']:.2f}     | {rep_pca['1']['precision']:.2f}               | {rep_pca['1']['recall']:.2f}            | {rep_pca['1']['f1-score']:.2f}")
print(f"{'SelectKBest':<15} | {rep_final['accuracy']:.2f}     | {rep_final['1']['precision']:.2f}               | {rep_final['1']['recall']:.2f}            | {rep_final['1']['f1-score']:.2f}")

--- 4. SUPPORT VECTOR MACHINE ---
Best PCA Params:   {'C': 1, 'class_weight': {0: 1, 1: 3}, 'kernel': 'rbf'}
Best Final Params: {'C': 1, 'class_weight': {0: 1, 1: 3}, 'kernel': 'rbf'}

Dataset         | Accuracy | Class 1 Precision  | Class 1 Recall  | Class 1 F1
--------------------------------------------------------------------------------
PCA             | 0.84     | 0.49               | 0.45            | 0.47
SelectKBest     | 0.82     | 0.45               | 0.47            | 0.46


In [ ]:
# --- 4. DECISION TREE A/B TEST ---
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

print("--- 4. DECISION TREE ---")
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_params = {"max_depth": [3, 5, 10], "min_samples_split": [2, 5]}

# 1. Train on PCA
grid_pca = GridSearchCV(dt, dt_params, cv=5, scoring='f1', n_jobs=-1)
grid_pca.fit(X_train_pca, y_train)
rep_pca = classification_report(y_test, grid_pca.predict(X_test_pca), output_dict=True)

# 2. Train on Final (SelectKBest)
grid_final = GridSearchCV(dt, dt_params, cv=5, scoring='f1', n_jobs=-1)
grid_final.fit(X_train_final, y_train)
rep_final = classification_report(y_test, grid_final.predict(X_test_final), output_dict=True)

# 3. Print Results
print(f"Best PCA Params:   {grid_pca.best_params_}")
print(f"Best Final Params: {grid_final.best_params_}\n")
print(f"{'Dataset':<15} | {'Accuracy':<8} | {'Class 1 Precision':<18} | {'Class 1 Recall':<15} | {'Class 1 F1'}")
print("-" * 80)
print(f"{'PCA':<15} | {rep_pca['accuracy']:.2f}     | {rep_pca['1']['precision']:.2f}               | {rep_pca['1']['recall']:.2f}            | {rep_pca['1']['f1-score']:.2f}")
print(f"{'SelectKBest':<15} | {rep_final['accuracy']:.2f}     | {rep_final['1']['precision']:.2f}               | {rep_final['1']['recall']:.2f}            | {rep_final['1']['f1-score']:.2f}")

--- 2. DECISION TREE ---
Best PCA Params:   {'max_depth': 5, 'min_samples_split': 2}
Best Final Params: {'max_depth': 3, 'min_samples_split': 2}

Dataset         | Accuracy | Class 1 Precision  | Class 1 Recall  | Class 1 F1
--------------------------------------------------------------------------------
PCA             | 0.74     | 0.33               | 0.62            | 0.43
SelectKBest     | 0.73     | 0.32               | 0.64            | 0.43


In [ ]:
# --- 5. RANDOM FOREST A/B TEST ---
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("--- 5. RANDOM FOREST ---")
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_params = {"n_estimators": [50, 100], "max_depth": [5, 10]}

# 1. Train on PCA
grid_pca = GridSearchCV(rf, rf_params, cv=5, scoring='f1', n_jobs=-1)
grid_pca.fit(X_train_pca, y_train)
rep_pca = classification_report(y_test, grid_pca.predict(X_test_pca), output_dict=True)

# 2. Train on Final (SelectKBest)
grid_final = GridSearchCV(rf, rf_params, cv=5, scoring='f1', n_jobs=-1)
grid_final.fit(X_train_final, y_train)
rep_final = classification_report(y_test, grid_final.predict(X_test_final), output_dict=True)

# 3. Print Results
print(f"Best PCA Params:   {grid_pca.best_params_}")
print(f"Best Final Params: {grid_final.best_params_}\n")
print(f"{'Dataset':<15} | {'Accuracy':<8} | {'Class 1 Precision':<18} | {'Class 1 Recall':<15} | {'Class 1 F1'}")
print("-" * 80)
print(f"{'PCA':<15} | {rep_pca['accuracy']:.2f}     | {rep_pca['1']['precision']:.2f}               | {rep_pca['1']['recall']:.2f}            | {rep_pca['1']['f1-score']:.2f}")
print(f"{'SelectKBest':<15} | {rep_final['accuracy']:.2f}     | {rep_final['1']['precision']:.2f}               | {rep_final['1']['recall']:.2f}            | {rep_final['1']['f1-score']:.2f}")

--- 3. RANDOM FOREST ---
Best PCA Params:   {'max_depth': 5, 'n_estimators': 100}
Best Final Params: {'max_depth': 5, 'n_estimators': 100}

Dataset         | Accuracy | Class 1 Precision  | Class 1 Recall  | Class 1 F1
--------------------------------------------------------------------------------
PCA             | 0.81     | 0.41               | 0.40            | 0.41
SelectKBest     | 0.81     | 0.42               | 0.55            | 0.48
